# Comparable training: ODRA, ODRA-native, and simulator ansätze

Trains three circuits aligned with [`models/model_odra.ipynb`](../../models/model_odra.ipynb), [`models/model_odra_native.ipynb`](../../models/model_odra_native.ipynb), and [`models/model_simulator.ipynb`](../../models/model_simulator.ipynb) on **identical** banknote data (same split and scaler as ODRA `prepare_data`), **same hyperparameters**, and **`StatevectorEstimator`**. Gradients use **`qiskit_machine_learning.gradients.ParamShiftEstimatorGradient`** (decomposes CRX/CRY via `TranslateParameterizedGates`); `qiskit_algorithms` reverse-mode gradient raises `KeyError` on the simulator ansatz.

**Entanglement (hybrid):** first block is a full **ring** — `cz(i, (i+1) mod n)` / `crx`. Second block is still a full **ring** (`n` two-qubit gates) but in **wrap-first walk order**: `(0, n−1)` then `(n−1, n−2), …, (1, 0)` — same edge *set* as the usual reverse ring, so parameter count matches the model notebooks (`4n` / `8n` per macro-layer), while the **last** gate targets **`q_0`** (aligned with **`MEASURE_QUBIT = 0`**). **ODRA** uses Ry / Rz+CZ / Rx / Ry+CZ; **ODRA-native** uses `R(θ,φ)` + CZ; **simulator** uses CRX / CRY.

**Readout:** Pauli **Z** on Qiskit qubit index **`MEASURE_QUBIT = 0`** (`q_0`). Pauli string convention: **rightmost character = qubit 0** (for 5 qubits, Z on `q_0` is `"IIIIZ"`).

**Outputs** (under this `setup/` folder):

- `test_data.csv` — scaled features `f0..f4` and binary labels `y` in `{0,1}`
- `weights_odra.csv`, `weights_odra_native.csv`, `weights_simulator.csv` — `param_index`, `param_name`, `value` (**best** test-accuracy checkpoint)
- `weights_*_best.pth`, `weights_*_final.pth` for each ansatz — full `state_dict` (**best** vs **last epoch**)

Ansatz depth is **2** (one macro-layer). At `N_QUBITS = 5`, **ODRA** and **simulator** each have **20** ansatz parameters (`4n`); **ODRA-native** has **40** (`8n`, two angles per `R`). Native is **not** parameter-matched to the other two; training conditions are otherwise shared.

**Reproducibility:** `RANDOM_SEED` fixes Python `random`, NumPy, PyTorch (CPU/CUDA), `train_test_split`, `DataLoader` shuffle (`Generator`), and each `train_and_save` call re-seeds before building the model. One shared `DataLoader` instance is used for all three runs so mini-batch order matches. CUDA uses deterministic cuDNN settings where applicable.


In [1]:
import csv
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import DataLoader, TensorDataset
from ucimlrepo import fetch_ucirepo

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.primitives import StatevectorEstimator
from qiskit.quantum_info import SparsePauliOp
from qiskit_machine_learning.connectors import TorchConnector
from qiskit_machine_learning.gradients import ParamShiftEstimatorGradient
from qiskit_machine_learning.neural_networks import EstimatorQNN

# --- Resolve repo root and output directory ---
ROOT = Path.cwd().resolve()
for parent in [ROOT, *ROOT.parents]:
    if (parent / "pyproject.toml").is_file():
        ROOT = parent
        break

SETUP_DIR = ROOT / "tests" / "ansatz_odra_evaluation" / "setup"
SETUP_DIR.mkdir(parents=True, exist_ok=True)

# --- Training / circuit config (match model_odra defaults; shared by all ansätze) ---
RANDOM_SEED = 42
N_QUBITS = 5
ANSATZ_DEPTH = 4
MEASURE_QUBIT = 0  # Z readout on q_0
EPOCHS = 30
BATCH_SIZE = 16
LEARNING_RATE = 0.01
# Suffix for weight filenames (empty here; trimmed notebook sets e.g. "_trimmed_reverse_q0")
ARTIFACT_SUFFIX = ""

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

print(f"ROOT={ROOT}")
print(f"SETUP_DIR={SETUP_DIR}")
print(f"Pauli Z on qubit index {MEASURE_QUBIT}, depth={ANSATZ_DEPTH}")


ROOT=/Users/jkw/Documents/uni/axion/QC1
SETUP_DIR=/Users/jkw/Documents/uni/axion/QC1/tests/ansatz_odra_evaluation/setup
Pauli Z on qubit index 0, depth=4


In [2]:
def prepare_data(test_size: float = 0.2, random_state: int = 42):
    # Same as models/model_odra.ipynb (returns train/test indices for reproducibility)
    banknote_authentication = fetch_ucirepo(id=267)
    X = banknote_authentication.data.features.to_numpy()
    y = banknote_authentication.data.targets.to_numpy().ravel()
    indices = np.arange(len(X))

    assert X.shape[1] == 4, f"Expected 4 features, got {X.shape[1]}"
    assert set(np.unique(y)) == {0, 1}, f"Expected binary labels {{0, 1}}, got {set(np.unique(y))}"

    variance = X[:, 0].reshape(-1, 1)
    skewness = X[:, 1].reshape(-1, 1)
    interaction = variance * skewness
    X_expanded = np.hstack((X, interaction))

    X_train, X_test, y_train, y_test, idx_train, idx_test = train_test_split(
        X_expanded, y, indices, test_size=test_size, random_state=random_state
    )

    scaler = MinMaxScaler(feature_range=(-np.pi / 4, np.pi / 4))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    print(f"Dataset loaded: {len(X_train)} train, {len(X_test)} test samples")
    print(f"Feature dimension: {X_train_scaled.shape[1]}")

    return X_train_scaled, X_test_scaled, y_train, y_test, idx_train, idx_test


def pauli_z_string(num_qubits: int, z_qubit_index: int) -> str:
    """Qiskit label: left q_{n-1} ... right q_0; Z on q_k at index (n-1-k) from left."""
    if not 0 <= z_qubit_index < num_qubits:
        raise ValueError(f"z_qubit_index must be in [0, {num_qubits}), got {z_qubit_index}")
    chars = ["I"] * num_qubits
    chars[num_qubits - 1 - z_qubit_index] = "Z"
    return "".join(chars)


X_train, X_test, y_train_raw, y_test_raw, train_idx, test_idx = prepare_data(
    test_size=0.2, random_state=RANDOM_SEED
)

pauli_label = pauli_z_string(N_QUBITS, MEASURE_QUBIT)
print(f"Observable Pauli string: {pauli_label!r}")


Dataset loaded: 1097 train, 275 test samples
Feature dimension: 5
Observable Pauli string: 'IIIIZ'


In [3]:
def odra_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    """Ry → forward ring → Rx → reverse ring in wrap-first order ending on q0."""
    params_per_iter = 4 * n_qubits
    theta = ParameterVector("theta", params_per_iter * (depth // 2))
    qc = QuantumCircuit(n_qubits)

    for j in range(depth // 2):
        offset = j * params_per_iter

        for i in range(n_qubits):
            qc.ry(theta[offset + i], i)

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            param_idx = offset + n_qubits + i
            qc.rz(theta[param_idx], target)
            qc.cz(control, target)

        offset_l2 = offset + 2 * n_qubits
        for i in range(n_qubits):
            qc.rx(theta[offset_l2 + i], i)

        base3 = offset_l2 + n_qubits
        qc.ry(theta[base3 + 0], n_qubits - 1)
        qc.cz(0, n_qubits - 1)
        for k in range(1, n_qubits):
            control = n_qubits - k
            target = n_qubits - k - 1
            qc.ry(theta[base3 + k], target)
            qc.cz(control, target)

    return qc


def odra_native_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    """Forward ring R+CZ; second block wrap-first ring walk (same edges as reverse ring)."""
    params_per_iter = 8 * n_qubits
    theta = ParameterVector("theta", params_per_iter * (depth // 2))
    qc = QuantumCircuit(n_qubits)

    for j in range(depth // 2):
        offset = j * params_per_iter
        p = offset

        for i in range(n_qubits):
            qc.r(theta[p], theta[p + 1], i)
            p += 2

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.r(theta[p], theta[p + 1], target)
            p += 2
            qc.cz(control, target)

        for i in range(n_qubits):
            qc.r(theta[p], theta[p + 1], i)
            p += 2

        qc.r(theta[p], theta[p + 1], n_qubits - 1)
        p += 2
        qc.cz(0, n_qubits - 1)
        for k in range(1, n_qubits):
            control = n_qubits - k
            target = n_qubits - k - 1
            qc.r(theta[p], theta[p + 1], target)
            p += 2
            qc.cz(control, target)

    return qc


def simulator_ansatz(n_qubits: int, depth: int) -> QuantumCircuit:
    """Forward ring CRX; second block CRY in wrap-first ring order (2*n*depth params)."""
    theta = ParameterVector("theta", 2 * n_qubits * depth)
    qc = QuantumCircuit(n_qubits)
    param_idx = 0

    for _ in range(depth // 2):
        for i in range(n_qubits):
            qc.ry(theta[param_idx], i)
            param_idx += 1

        for i in range(n_qubits):
            control = i
            target = (i + 1) % n_qubits
            qc.crx(theta[param_idx], control, target)
            param_idx += 1

        for i in range(n_qubits):
            qc.rx(theta[param_idx], i)
            param_idx += 1

        qc.cry(theta[param_idx], 0, n_qubits - 1)
        param_idx += 1
        for k in range(1, n_qubits):
            control = n_qubits - k
            target = n_qubits - k - 1
            qc.cry(theta[param_idx], control, target)
            param_idx += 1

    return qc


In [4]:
# --- Verify: forward ring + second-layer wrap-first ring walk (per macro-layer) ---
# Tuples are (control_qubit, target_qubit) in circuit order.


def ring_entangle_edges_forward(n: int) -> list[tuple[int, int]]:
    """First entangling block: full ring (i, (i+1) mod n)."""
    return [(i, (i + 1) % n) for i in range(n)]


def ring_second_layer_wrap_first_toward_q0(n: int) -> list[tuple[int, int]]:
    """Second block: wrap (0, n−1) then (n−1,n−2)…(1,0); n edges, same set as reverse ring."""
    edges = [(0, n - 1)]
    for k in range(1, n):
        edges.append((n - k, n - k - 1))
    return edges


print("Edge convention: (control_qubit, target_qubit)")
print(f"Layer 1 — forward ring: {ring_entangle_edges_forward(N_QUBITS)}")
print(f"Layer 2 — wrap-first ring (ends on q_0): {ring_second_layer_wrap_first_toward_q0(N_QUBITS)}")
_macro = ANSATZ_DEPTH // 2
if _macro != 1:
    print(f"(Each macro-layer repeats those two blocks; depth={ANSATZ_DEPTH} → {_macro} repetitions.)")
print()

for _label, _fn in [
    ("ODRA", odra_ansatz),
    ("ODRA-native", odra_native_ansatz),
    ("simulator", simulator_ansatz),
]:
    _qc = _fn(N_QUBITS, ANSATZ_DEPTH)
    print("=" * 72)
    print(f"{_label}  |  n={N_QUBITS}, depth={ANSATZ_DEPTH}  |  ansatz parameters: {len(_qc.parameters)}")
    print(_qc.draw(fold=-1))
    print(f"  Entangling layer 1 (ring): {ring_entangle_edges_forward(N_QUBITS)}")
    print(f"  Entangling layer 2 (wrap-first ring): {ring_second_layer_wrap_first_toward_q0(N_QUBITS)}")
    print()


Edge convention: (control_qubit, target_qubit)
Layer 1 — forward ring: [(0, 1), (1, 2), (2, 3), (3, 4), (4, 0)]
Layer 2 — wrap-first ring (ends on q_0): [(0, 4), (4, 3), (3, 2), (2, 1), (1, 0)]
(Each macro-layer repeats those two blocks; depth=4 → 2 repetitions.)

ODRA  |  n=5, depth=4  |  ansatz parameters: 40
     ┌──────────────┐                   ┌──────────────┐                                     ┌───────────────┐                    ┌───────────────┐                                                   ┌───────────────┐                    ┌───────────────┐                                     ┌───────────────┐                    ┌───────────────┐         
q_0: ┤ Ry(theta[0]) ├─────────────────■─┤ Rz(theta[9]) ├───────────────────────────────────■─┤ Rx(theta[10]) ├──────────────────■─┤ Ry(theta[19]) ├──────────────────────────────────────────■────────┤ Ry(theta[20]) ├──────────────────■─┤ Rz(theta[29]) ├───────────────────────────────────■─┤ Rx(theta[30]) ├──────────────────■─┤ Ry(the

In [5]:
class HybridModel(nn.Module):
    """VQC with configurable Z-observable qubit (same structure as model notebooks)."""

    def __init__(self, ansatz_circuit: QuantumCircuit, num_qubits: int, z_qubit_index: int):
        super().__init__()
        self.feature_map = self._create_angle_encoding(num_qubits)
        self.qc = QuantumCircuit(num_qubits)
        self.qc.compose(self.feature_map, qubits=range(num_qubits), inplace=True)
        self.qc.compose(ansatz_circuit, inplace=True)

        input_params = list(self.feature_map.parameters)
        weight_params = list(ansatz_circuit.parameters)

        obs_str = pauli_z_string(num_qubits, z_qubit_index)
        observable = SparsePauliOp.from_list([(obs_str, 1.0)])

        estimator = StatevectorEstimator()
        # QML ParamShift + TranslateParameterizedGates decomposes CRX/CRY; qiskit-algorithms
        # ReverseEstimatorGradient hits KeyError on simulator ansatz (gradient_parameter_map).
        gradient = ParamShiftEstimatorGradient(estimator=estimator)
        self.qnn = EstimatorQNN(
            circuit=self.qc,
            observables=observable,
            input_params=input_params,
            weight_params=weight_params,
            estimator=estimator,
            gradient=gradient,
        )
        self.quantum_layer = TorchConnector(self.qnn)

    def _create_angle_encoding(self, num_qubits: int) -> QuantumCircuit:
        qc_data = QuantumCircuit(num_qubits)
        input_params = ParameterVector("x", num_qubits)
        for i in range(num_qubits):
            qc_data.ry(input_params[i], i)
        return qc_data

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.quantum_layer(x)


In [6]:
# Labels in {-1, +1} for MSE vs Z expectation; features float32
y_train = 2 * y_train_raw - 1
y_test_pm = 2 * y_test_raw - 1

X_train = X_train.astype(np.float32)
X_test = X_test.astype(np.float32)

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).reshape(-1, 1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_pm, dtype=torch.float32).reshape(-1, 1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)


def make_train_loader():
    g = torch.Generator()
    g.manual_seed(RANDOM_SEED)
    return DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=g)


loss_function = nn.MSELoss()


def train_epoch(model, train_loader, optimizer, loss_fn):
    model.train()
    epoch_loss = 0.0
    num_batches = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        num_batches += 1
    return epoch_loss / num_batches


def evaluate(model, X_tensor, y_tensor, loss_fn):
    model.eval()
    with torch.no_grad():
        outputs = model(X_tensor)
        loss = loss_fn(outputs, y_tensor).item()
        predicted = (outputs > 0).float() * 2 - 1
        correct = (predicted == y_tensor).sum().item()
        accuracy = correct / len(y_tensor)
    return {"loss": loss, "accuracy": accuracy}


def save_test_csv():
    path = SETUP_DIR / "test_data.csv"
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["f0", "f1", "f2", "f3", "f4", "y"])
        for row, lab in zip(X_test, y_test_raw):
            w.writerow([float(row[0]), float(row[1]), float(row[2]), float(row[3]), float(row[4]), int(lab)])
    print(f"Wrote {path}")


def save_weights_csv(model, ansatz_circuit, path: Path):
    vals = model.quantum_layer.weight.detach().cpu().numpy().reshape(-1)
    params = list(ansatz_circuit.parameters)
    assert len(vals) == len(params), (len(vals), len(params))
    with open(path, "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["param_index", "param_name", "value"])
        for i, (p, v) in enumerate(zip(params, vals)):
            w.writerow([i, str(p), float(v)])
    print(f"Wrote {path}")


save_test_csv()


Wrote /Users/jkw/Documents/uni/axion/QC1/tests/ansatz_odra_evaluation/setup/test_data.csv


In [ ]:
def train_and_save(name: str, ansatz_circuit: QuantumCircuit, loader):
    random.seed(RANDOM_SEED)
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(RANDOM_SEED)
    gen = getattr(loader, "generator", None)
    if gen is not None:
        gen.manual_seed(RANDOM_SEED)

    model = HybridModel(ansatz_circuit, N_QUBITS, MEASURE_QUBIT)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    best_acc = 0.0
    best_state = None

    print(f"\n=== Training {name} ===")
    print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad)}")

    for epoch in range(EPOCHS):
        train_loss = train_epoch(model, loader, optimizer, loss_function)
        test_metrics = evaluate(model, X_test_tensor, y_test_tensor, loss_function)
        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(
                f"Epoch {epoch + 1:3d}/{EPOCHS} | Train Loss: {train_loss:.4f} | "
                f"Test Acc: {test_metrics['accuracy']:.4f}"
            )
        if test_metrics["accuracy"] > best_acc:
            best_acc = test_metrics["accuracy"]
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    stem = name.lower().replace(" ", "_")
    torch.save(model.state_dict(), SETUP_DIR / f"weights_{stem}_final{ARTIFACT_SUFFIX}.pth")

    if best_state is not None:
        model.load_state_dict(best_state)
    print(f"Best test accuracy ({name}): {best_acc:.4f}")

    save_weights_csv(model, ansatz_circuit, SETUP_DIR / f"weights_{stem}{ARTIFACT_SUFFIX}.csv")
    torch.save(model.state_dict(), SETUP_DIR / f"weights_{stem}_best{ARTIFACT_SUFFIX}.pth")
    return model


odra_q = odra_ansatz(N_QUBITS, ANSATZ_DEPTH)
odra_native_q = odra_native_ansatz(N_QUBITS, ANSATZ_DEPTH)
sim_q = simulator_ansatz(N_QUBITS, ANSATZ_DEPTH)

shared_train_loader = make_train_loader()

train_and_save("odra", odra_q, shared_train_loader)
#train_and_save("odra_native", odra_native_q, shared_train_loader)
train_and_save("simulator", sim_q, shared_train_loader)

print("\nDone. Artifacts in:", SETUP_DIR)



=== Training odra ===
Trainable parameters: 40
Epoch   1/30 | Train Loss: 0.7693 | Test Acc: 0.7927
